# Step 11 — Answer the six business questions

This notebook runs the same six SQL business questions against the completed ETL and ELT September warehouses. Each query is stored in a checked-in SQL file, and every result is displayed and exported for submission.

The notebook uses read-only warehouse connections and does not edit source data or the read-only snapshots.

In [ ]:
# Import the SQL runner and configure paths to the Step 9 warehouse artifacts.
from pathlib import Path
import sys

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'include').exists():
    REPO_ROOT = Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT / 'include'))

from business_questions import (
    explain_september_difference,
    run_business_questions,
    september_revenue,
)

ETL_DB = REPO_ROOT / 'assignment' / 'warehouse_samy_etl_step9.duckdb'
ELT_DB = REPO_ROOT / 'assignment' / 'warehouse_samy_elt_step9.duckdb'
assert ETL_DB.exists(), f'Missing ETL warehouse: {ETL_DB}'
assert ELT_DB.exists(), f'Missing ELT warehouse: {ELT_DB}'
print(f'ETL warehouse: {ETL_DB}')
print(f'ELT warehouse: {ELT_DB}')

In [ ]:
# Execute all six queries for each pipeline. The SQL files contain one
# named query per business question, so results remain easy to audit.
etl_results = run_business_questions(ETL_DB, 'ETL')
elt_results = run_business_questions(ELT_DB, 'ELT')

for question_name in etl_results:
    print(f'\n=== {question_name} — ETL ===')
    display(etl_results[question_name])
    print(f'=== {question_name} — ELT ===')
    display(elt_results[question_name])

In [ ]:
# Compare the like-for-like September delivered net-revenue totals.
# If they differ, list the exact order IDs included by only one pipeline.
etl_total = september_revenue(ETL_DB, 'ETL')
elt_total = september_revenue(ELT_DB, 'ELT')
difference = etl_total - elt_total
print(f'ETL September net revenue: {etl_total:,.2f}')
print(f'ELT September net revenue: {elt_total:,.2f}')
print(f'Difference (ETL - ELT): {difference:,.2f}')

if abs(difference) < 0.01:
    print('September totals match within one cent.')
else:
    differences = explain_september_difference(ETL_DB, ELT_DB)
    print('Totals differ. The following order IDs explain the difference:')
    display(differences)
    print('Interpretation: ETL excludes rows failing critical Python checks before the fact table; ELT applies its SQL staging rules and therefore may retain a different set of passing rows.')

In [ ]:
# Export every result so the SQL answers can be attached to the submission.
output_dir = REPO_ROOT / 'assignment' / 'step11_business_questions'
output_dir.mkdir(parents=True, exist_ok=True)
for pipeline_name, results in [('etl', etl_results), ('elt', elt_results)]:
    for question_name, frame in results.items():
        frame.to_csv(output_dir / f'{pipeline_name}_{question_name}.csv', index=False)

comparison = {
    'etl_september_net_revenue': etl_total,
    'elt_september_net_revenue': elt_total,
    'difference_etl_minus_elt': difference,
}
import json
(output_dir / 'september_total_comparison.json').write_text(json.dumps(comparison, indent=2))
print(f'Exported {len(etl_results) + len(elt_results)} result CSV files to {output_dir}')
print('Exported September total comparison to september_total_comparison.json')

## Business-question mapping

1. Daily net revenue by zone and store → `etl_daily_zone_revenue` / `mart_daily_store_kpis`.
2. Revenue and gross margin by category → `etl_category_margin` / `mart_category_margin`.
3. Average delivery time and percentage within 15 minutes → ETL fact calculation / `mart_delivery_sla`.
4. Revenue by payment mode and unreconciled amount → ETL fact plus reconciliation / staging plus `mart_payment_reconciliation`.
5. Top 10 customers by spend and loyalty tier → ETL dimensions plus fact / `mart_top_customers`.
6. Rejected rows by file and failure reason → ETL `dq_results` / ELT staging `dq_status` and `failure_reason`.